# AI-Powered Cardiac Risk Assessment and Heart Failure Prediction

**IEEE-aligned research notebook** (Jupyter / Google Colab)

This notebook reproduces and extends *Prognostic Modeling for Heart Failure Survival: A Classification Approach* (Sandilya et al., SPIN 2024, IEEE) and the submitted abstract on stacking / XGBoost heart-failure prediction.

### Pipeline (SPIN 2024 Fig. 1 + extensions)
1. Data preprocessing and StandardScaler standardization  
2. Classification: Naive Bayes, KNN, Decision Tree, SVM, Logistic Regression, Random Forest, XGBoost  
3. **Extension:** Stacking Classifier (meta-learner = Logistic Regression; base-learners = XGBoost, Random Forest, SVM)  
4. GridSearchCV hyperparameter tuning (accuracy, F1, MCC, ROC-AUC)  
5. Evaluation matrices and feature-importance ranking  
6. Multi-modal ECG interval checks and waveform visualization  

> **Disclaimer:** Educational / academic prototype only. Not a clinical diagnostic device.

## 0. Environment
Install the research stack, then make the project root importable. In Colab, upload or clone this repository so `config.py` sits next to this notebook.

In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "config.py").exists():
    for cand in ROOT.rglob("config.py"):
        ROOT = cand.parent
        break
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

pkgs = [
    "numpy", "pandas", "scipy", "scikit-learn", "xgboost",
    "imbalanced-learn", "matplotlib", "seaborn", "joblib",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Dependencies ready.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay

import config
from data_loader import DatasetHandler
from preprocessing import DataPreprocessor
from models import AdvancedHeartModels
from pipeline import CardiacAssessmentPipeline, synthesize_ecg_waveform

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
print("IEEE schema features:", len(config.ALL_FEATURES))
print("Priority predictors (SPIN 2024 Fig. 2):", config.PRIORITY_FEATURES)

## 1. Dataset specification
The baseline UCI Heart Failure Clinical Records corpus contains **299 patients** and **12 clinical attributes** plus `DEATH_EVENT` (SPIN 2024 Table I). This project expands that schema to **> 20,000 synthetic records** using:

- a **Gaussian copula** (CTGAN-style joint tabular sampler) fitted on a physiologically constrained seed cohort;
- **SMOTE**-style interpolation of the minority mortality class;
- synthetic **ECG interval columns** (PR, QRS, QT, QTc, ST deviation, RR, P-wave, T-wave amplitude, heart rate) and derived conduction flags.

Labels are generated so **ejection fraction, serum creatinine, follow-up time, and age** remain the dominant risk drivers, matching the base paper.

In [ ]:
handler = DatasetHandler(random_state=config.RANDOM_STATE)
df = handler.load_or_generate(n_records=config.TARGET_SYNTHETIC_RECORDS, persist=True)
summary = handler.summarize(df)
display(pd.Series(summary))
df.head()

In [ ]:
assert len(df) > 20_000, "Cohort must exceed 20,000 records."
print(f"Records: {len(df):,} | Event rate: {df[config.TARGET_COLUMN].mean():.3f}")
print("ECG columns:", config.ECG_FEATURES)
df[config.CORE_FEATURES + [config.TARGET_COLUMN]].describe().T

## 2. Exploratory analysis and class imbalance
SPIN 2024 notes that the original 299-row set is imbalanced. We inspect the expanded cohort before StandardScaler + SMOTE.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df[config.TARGET_COLUMN].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#1f9d6a", "#c81e1e"]
)
axes[0].set_title("DEATH_EVENT class counts")
axes[0].set_xticklabels(["Survival (0)", "Death (1)"], rotation=0)

sns.kdeplot(data=df, x="ejection_fraction", hue=config.TARGET_COLUMN, ax=axes[1], fill=True)
axes[1].set_title("Ejection fraction by outcome")
sns.kdeplot(data=df, x="serum_creatinine", hue=config.TARGET_COLUMN, ax=axes[2], fill=True)
axes[2].set_title("Serum creatinine by outcome")
plt.tight_layout()
plt.show()

## 3. Preprocessing (impute → StandardScaler → SMOTE)
Following SPIN 2024 Section IV, numerical attributes are standardized to zero mean / unit variance so scale disparities (platelets vs. creatinine) do not dominate distance-based models (KNN, SVM).

In [ ]:
pre = DataPreprocessor()
cleaned = pre.handle_missing(df)
X, y = pre.split_xy(cleaned)
print("Feature matrix:", X.shape, "| positives:", int(y.sum()))
X.isna().sum().sum()

## 4. Train / evaluate the full classifier suite
Training uses an 80/20 stratified split (SPIN 2024). For Colab runtime limits, GridSearch uses the **fast** grids in `config.FAST_PARAM_GRIDS` (paper-scale grids remain in `config.PAPER_PARAM_GRIDS`). Stacking is always trained after base learners.

In [ ]:
pipe = CardiacAssessmentPipeline(fast_tuning=True)
result = pipe.run(df=df, tune=True, persist=True, max_train_rows=4000)
metrics = result.metrics.copy()
display(metrics.round(4))
print("Best model:", result.models.best_model_name)
print("Train size after SMOTE:", result.n_train, "| Test size:", result.n_test)

## 5. IEEE evaluation matrix
Metrics correspond to SPIN 2024 Tables II–IV, extended with precision/recall and the stacking ensemble from the submitted abstract.

In [ ]:
ieee = result.models.export_ieee_metrics_table()
display(ieee.round(4))

fig, ax = plt.subplots(figsize=(10, 4.5))
plot_df = ieee.set_index("model")[["test_accuracy", "test_f1", "test_mcc", "test_roc_auc", "test_pr_auc"]]
plot_df.plot(kind="bar", ax=ax, colormap="Reds")
ax.set_ylim(0, 1.05)
ax.set_title("Hold-out performance (IEEE metric suite)")
ax.legend(loc="lower right", ncol=3)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
best = result.models.best_model_name
X_all, y_all = result.preprocessor.split_xy(cleaned)
# Reconstruct the same split used internally is not required for a qualitative CM on a hold-out redraw.
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=config.RANDOM_STATE
)
X_te_s = result.preprocessor.transform(X_te)
y_pred = result.models.models[best].predict(X_te_s)
y_score = result.models.models[best].predict_proba(X_te_s)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
ConfusionMatrixDisplay.from_predictions(y_te, y_pred, ax=axes[0], cmap="Reds")
axes[0].set_title(f"Confusion matrix — {best}")
RocCurveDisplay.from_predictions(y_te, y_score, ax=axes[1])
axes[1].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_te, y_score, ax=axes[2])
axes[2].set_title("Precision–Recall curve")
plt.tight_layout()
plt.show()

## 6. Feature importance (SPIN 2024 Fig. 2)
The base paper ranks **time, ejection fraction, serum creatinine, and age** as the four most important attributes. The plot below uses XGBoost gain on the expanded multi-modal feature set.

In [ ]:
fi = result.importances.copy()
display(fi.head(12))
fig, ax = plt.subplots(figsize=(8, 6))
top = fi.head(12)
ax.barh(top["feature"][::-1], top["importance"][::-1], color="#c81e1e")
ax.set_title("XGBoost feature importance")
ax.set_xlabel("Relative importance")
plt.tight_layout()
plt.show()
print("Priority features in ranking:")
for feat in config.PRIORITY_FEATURES:
    rank = list(fi["feature"]).index(feat) + 1
    print(f"  {feat:22s} rank {rank}")

## 7. Multi-modal ECG diagnostic visualizer
Synthetic lead-II morphology is reconstructed from PR, QRS, QT, ST, and heart-rate parameters. Flags highlight conduction delay, QTc prolongation, ST shift, bradycardia, and tachycardia — complementary to heart-failure survival prediction.

In [ ]:
sample = df.iloc[0]
t, wave = synthesize_ecg_waveform(
    pr_interval_ms=sample.pr_interval_ms,
    qrs_duration_ms=sample.qrs_duration_ms,
    qt_interval_ms=sample.qt_interval_ms,
    st_deviation_mm=sample.st_deviation_mm,
    heart_rate_bpm=sample.heart_rate_bpm,
    p_wave_ms=sample.p_wave_ms,
)
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(t, wave, color="#1f9d6a", lw=1.2)
ax.set_title("Synthetic lead-II ECG from interval features")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (a.u.)")
ax.set_facecolor("#0b1220")
fig.patch.set_facecolor("#0b1220")
ax.tick_params(colors="white")
ax.xaxis.label.set_color("white")
ax.yaxis.label.set_color("white")
ax.title.set_color("white")
plt.tight_layout()
plt.show()
print(sample[config.DERIVED_ECG_FLAGS].to_dict())

## 8. Single-patient inference
The persisted `artifacts/cardiac_risk_bundle.joblib` is the same artifact consumed by the Streamlit dashboard (`streamlit run app.py`).

In [ ]:
patient = cleaned.iloc[[12]][result.models.feature_names]
X_inf = result.preprocessor.process_inference_matrix(patient)
proba = result.models.predict_proba_best(X_inf)[0, 1]
print("Mortality probability during follow-up: {:.1%}".format(proba))
print("Thresholded label:", int(proba >= 0.5))
print("Bundle:", result.bundle_path)

## References
1. P. K. Sandilya et al., “Prognostic Modeling for Heart Failure Survival: A Classification Approach,” SPIN 2024, IEEE.  
2. D. Chicco and G. Jurman, BMC Med. Inform. Decis. Mak., 2020.  
3. T. Ahmad et al., PLOS ONE, 2017. UCI Heart Failure Clinical Records.  
4. T. Chen and C. Guestrin, “XGBoost,” KDD 2016.